# Notebook 21 — Análisis comparativo: Config A vs C10 vs SAM2

**TFM — Sistema de Detección de Amenazas Armadas en Vídeo**  
Oliver Legarreta García · Universitat Oberta de Catalunya

---

Este notebook presenta el análisis comparativo completo de las tres estrategias de preprocesado evaluadas sobre el dataset GAR (258 clips). No requiere GPU.

| Config | Estrategia | Modelos |
|--------|-----------|--------|
| **A** | Frame completo (baseline) | weapon_model |
| **C10** | BBox persona + 10% padding | seg_model + weapon_model |
| **SAM2** | BBox persona + SAM2 por objeto | seg_model + SAM2 + weapon_model |

**Modelo de armas:** `yolov8m_weapons_B_e50_640` · CONF=0.25 · Umbral clip=5 frames  
**Dataset:** GAR test set — 140 clips positivos + 118 clips negativos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install matplotlib pandas seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# ── Rutas ──────────────────────────────────────────────────────────────────────
SEG_DIR  = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation'
SAM2_DIR = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/sam2_ablation'
OUT_DIR  = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/comparativa_final'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Estilo ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
})

CFG_COLORS = {'A': '#4C72B0', 'C10': '#55A868', 'SAM2': '#DD8452'}
CFG_LABELS = {'A': 'A — Sin seg (baseline)', 'C10': 'C10 — BBox +10%', 'SAM2': 'SAM2 — Obj. aislado'}

CAT_LABELS = {
    'N1':'Walking',       'N2':'Jogging',
    'N3':'Running',       'N4':'Sneaking',
    'N5':'Phone relaxed', 'N6':'Phone looking',
    'N7':'Phone 2h',      'N8':'Phone rec 1h',
    'N9':'Phone rec 2h',  'N10':'Bottle',
    'N11':'Drinking',     'N12':'Heavy obj',
}

# ── Métricas globales (de los TXTs de evaluación) ─────────────────────────────
METRICS = {
    'A':    {'mAP50': 0.7789, 'mAP5095': 0.2674, 'Accuracy': 0.7519,
             'Precision': 0.7209, 'Recall': 0.8857, 'F1': 0.7949,
             'TP': 124, 'TN': 70, 'FP': 48, 'FN': 16},
    'C10':  {'mAP50': 0.7947, 'mAP5095': 0.2710, 'Accuracy': 0.7519,
             'Precision': 0.7407, 'Recall': 0.8571, 'F1': 0.7947,
             'TP': 120, 'TN': 76, 'FP': 42, 'FN': 20},
    'SAM2': {'mAP50': 0.8354, 'mAP5095': 0.3630, 'Accuracy': 0.6434,
             'Precision': 0.6200, 'Recall': 0.8857, 'F1': 0.7294,
             'TP': 124, 'TN': 42, 'FP': 76, 'FN': 16},
}

# ── Cargar CSVs ────────────────────────────────────────────────────────────────
dfs = {
    'A':    pd.read_csv(f'{SEG_DIR}/clip_results_A_sin_seg.csv'),
    'C10':  pd.read_csv(f'{SEG_DIR}/clip_results_C10_bbox_pad10.csv'),
    'SAM2': pd.read_csv(f'{SAM2_DIR}/clip_results_SAM2.csv'),
}
for cfg, df in dfs.items():
    df['true'] = df['true'].astype(int)
    df['pred'] = df['pred'].astype(int)

# FP% por categoría
cats = [f'N{i}' for i in range(1, 13)]
fp_pct = {}
for cfg, df in dfs.items():
    neg = df[df['true'] == 0]
    fp_pct[cfg] = {}
    for cat in cats:
        sub = neg[neg['category'] == cat]
        fp_pct[cfg][cat] = sub['pred'].sum() / len(sub) * 100 if len(sub) > 0 else 0

print('✅ Datos cargados')
for cfg, m in METRICS.items():
    print(f'   {cfg:<6}: F1={m["F1"]:.4f}  Prec={m["Precision"]:.4f}  '
          f'Rec={m["Recall"]:.4f}  FP={m["FP"]}  FN={m["FN"]}')

---
## 1. Tabla de métricas globales

In [ ]:
cfgs = ['A', 'C10', 'SAM2']
keys_frame = ['mAP50', 'mAP5095']
keys_clip  = ['Accuracy', 'Precision', 'Recall', 'F1']
keys_conf  = ['TP', 'TN', 'FP', 'FN']

col_w = 10
header = f"  {'Métrica':<16}" + ''.join(f'{c:>{col_w}}' for c in cfgs) + '  mejor'
sep    = '  ' + '-' * (16 + col_w * 3 + 8)

for section, keys, is_float, higher in [
    ('Frame-level', keys_frame, True, True),
    ('Clip-level',  keys_clip,  True, True),
    ('Confusión',   keys_conf,  False, None),
]:
    print(f'  ═══ {section} ═══')
    print(header); print(sep)
    for k in keys:
        vals = [METRICS[c][k] for c in cfgs]
        row  = f"  {k:<16}" + ''.join(
            f'{v:>{col_w}.4f}' if is_float else f'{int(v):>{col_w}}' for v in vals
        )
        if is_float:
            hib = k not in ['FP', 'FN']
            best = cfgs[int(np.argmax(vals) if hib else np.argmin(vals))]
            row += f'   {best}'
        print(row)
    print()

---
## 2. Figura 1 — Métricas clip-level comparadas

In [ ]:
met_keys = ['Accuracy', 'Precision', 'Recall', 'F1']
x = np.arange(len(met_keys))
w = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, cfg in enumerate(cfgs):
    vals   = [METRICS[cfg][k] for k in met_keys]
    offset = (i - 1) * w
    bars   = ax.bar(x + offset, vals, w, label=CFG_LABELS[cfg],
                    color=CFG_COLORS[cfg], alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.004,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)

# Línea de referencia baseline F1
ax.axhline(METRICS['A']['F1'], color=CFG_COLORS['A'],
           linestyle='--', alpha=0.5, linewidth=1.2, label='F1 baseline (A)')

ax.set_xticks(x)
ax.set_xticklabels(met_keys, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Valor')
ax.set_title('Métricas clip-level — Config A vs C10 vs SAM2')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig1_metricas_clip.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Figura 2 — FP y FN absolutos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# FP
ax = axes[0]
fp_vals = [METRICS[c]['FP'] for c in cfgs]
bars = ax.bar(cfgs, fp_vals, color=[CFG_COLORS[c] for c in cfgs], alpha=0.85, width=0.5)
for bar, v in zip(bars, fp_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.5, str(v),
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Falsos positivos totales')
ax.set_ylabel('Nº de clips')
ax.set_ylim(0, 100)
ax.axhline(METRICS['A']['FP'], color=CFG_COLORS['A'],
           linestyle='--', alpha=0.5, linewidth=1)

# FN
ax = axes[1]
fn_vals = [METRICS[c]['FN'] for c in cfgs]
bars = ax.bar(cfgs, fn_vals, color=[CFG_COLORS[c] for c in cfgs], alpha=0.85, width=0.5)
for bar, v in zip(bars, fn_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.3, str(v),
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Falsos negativos totales')
ax.set_ylabel('Nº de clips')
ax.set_ylim(0, 30)
ax.axhline(METRICS['A']['FN'], color=CFG_COLORS['A'],
           linestyle='--', alpha=0.5, linewidth=1)

plt.suptitle('FP y FN totales — 118 negativos / 140 positivos', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig2_fp_fn_totales.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Figura 3 — FP% por categoría negativa

In [ ]:
x = np.arange(len(cats))
w = 0.25

fig, ax = plt.subplots(figsize=(16, 6))
for i, cfg in enumerate(cfgs):
    vals   = [fp_pct[cfg][cat] for cat in cats]
    offset = (i - 1) * w
    ax.bar(x + offset, vals, w, label=CFG_LABELS[cfg],
           color=CFG_COLORS[cfg], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(
    [f'{c}\n{CAT_LABELS[c]}' for c in cats], fontsize=9
)
ax.set_ylabel('Tasa de falsos positivos (%)')
ax.set_ylim(0, 110)
ax.set_title('FP% por categoría negativa — Config A vs C10 vs SAM2')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig3_fp_por_categoria.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Figura 4 — Heatmap de FP% por categoría

In [ ]:
# Matriz para el heatmap
hm_data = pd.DataFrame(
    {cfg: [fp_pct[cfg][cat] for cat in cats] for cfg in cfgs},
    index=[f'{c} — {CAT_LABELS[c]}' for c in cats]
)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    hm_data, annot=True, fmt='.0f', cmap='RdYlGn_r',
    vmin=0, vmax=100, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'FP%'}
)
ax.set_xticklabels([CFG_LABELS[c] for c in cfgs], fontsize=9)
ax.set_title('FP% por categoría negativa — heatmap')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig4_heatmap_fp.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Figura 5 — Transiciones respecto al baseline (Config A)

In [ ]:
def get_transitions(df_a, df_other):
    m = df_a[['clip','true','pred','category']].merge(
        df_other[['clip','pred']], on='clip', suffixes=('_A','_B')
    )
    counts = {'FP→TN ✅': 0, 'TN→FP ❌': 0, 'TP→FN ❌': 0, 'FN→TP ✅': 0}
    for _, r in m.iterrows():
        t, a, b = int(r['true']), int(r['pred_A']), int(r['pred_B'])
        if   t==0 and a==1 and b==0: counts['FP→TN ✅'] += 1
        elif t==0 and a==0 and b==1: counts['TN→FP ❌'] += 1
        elif t==1 and a==1 and b==0: counts['TP→FN ❌'] += 1
        elif t==1 and a==0 and b==1: counts['FN→TP ✅'] += 1
    return counts

trans = {
    'C10':  get_transitions(dfs['A'], dfs['C10']),
    'SAM2': get_transitions(dfs['A'], dfs['SAM2']),
}

t_keys  = ['FP→TN ✅', 'TN→FP ❌', 'TP→FN ❌', 'FN→TP ✅']
t_colors = ['#2ecc71', '#e74c3c', '#e74c3c', '#2ecc71']
t_alpha  = [0.85, 0.85, 0.5, 0.5]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cfg in zip(axes, ['C10', 'SAM2']):
    vals = [trans[cfg][k] for k in t_keys]
    bars = ax.bar(t_keys, vals,
                  color=t_colors, alpha=0.85, width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.3, str(v),
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_title(f'Transiciones A → {cfg}', fontsize=12)
    ax.set_ylabel('Nº de clips')
    ax.set_ylim(0, max(max(trans['C10'].values()), max(trans['SAM2'].values())) + 8)
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle('Cambios de clasificación respecto al baseline (Config A)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig5_transiciones.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla resumen
print(f'  {"Transición":<15}  {"C10":>6}  {"SAM2":>6}  Interpretación')
print('  ' + '-'*60)
interp = {
    'FP→TN ✅': 'FP corregidos (mejora)',
    'TN→FP ❌': 'FP nuevos introducidos (empeora)',
    'TP→FN ❌': 'TP perdidos (empeora recall)',
    'FN→TP ✅': 'FN recuperados (mejora recall)',
}
for k in t_keys:
    print(f'  {k:<15}  {trans["C10"][k]:>6}  {trans["SAM2"][k]:>6}  {interp[k]}')

---
## 7. Figura 6 — Detección por frames: distribución de det_frames

In [ ]:
# Comparar cuántos frames con arma detecta cada config en positivos
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, cfg in zip(axes, cfgs):
    pos = dfs[cfg][dfs[cfg]['true'] == 1].copy()
    pos['det_pct'] = pos['det_frames'].astype(int) / pos['total_frames'].astype(int) * 100
    correct = pos[pos['pred'] == 1]['det_pct']
    missed  = pos[pos['pred'] == 0]['det_pct']
    ax.hist(correct, bins=20, alpha=0.7, color=CFG_COLORS[cfg],
            label=f'TP ({len(correct)})', density=False)
    ax.hist(missed, bins=20, alpha=0.7, color='#e74c3c',
            label=f'FN ({len(missed)})', density=False)
    ax.axvline(5/pos['total_frames'].astype(int).mean()*100,
               color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_title(f'Config {cfg}', fontsize=11)
    ax.set_xlabel('% frames con arma detectada')
    ax.set_ylabel('Nº de clips')
    ax.legend(fontsize=9)

plt.suptitle('Distribución de detecciones por frame en clips positivos', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig6_dist_detframes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Figura 7 — SAM2: impacto del 'sin persona' en la detección

In [ ]:
# Solo disponible en SAM2
sam2 = dfs['SAM2'].copy()
sam2['no_person'] = sam2['no_person'].astype(int)
sam2['total_frames'] = sam2['total_frames'].astype(int)
sam2['det_frames'] = sam2['det_frames'].astype(int)
sam2['no_person_pct'] = sam2['no_person'] / sam2['total_frames'] * 100
sam2['det_pct'] = sam2['det_frames'] / sam2['total_frames'] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: % sin persona vs % frames con arma (positivos)
ax = axes[0]
pos_sam2 = sam2[sam2['true'] == 1]
tp = pos_sam2[pos_sam2['pred'] == 1]
fn = pos_sam2[pos_sam2['pred'] == 0]
ax.scatter(tp['no_person_pct'], tp['det_pct'], color='#2ecc71',
           alpha=0.7, s=40, label=f'TP ({len(tp)})')
ax.scatter(fn['no_person_pct'], fn['det_pct'], color='#e74c3c',
           alpha=0.9, s=60, marker='X', label=f'FN ({len(fn)})')
ax.set_xlabel('% frames sin persona detectada')
ax.set_ylabel('% frames con arma detectada')
ax.set_title('Clips positivos — fallback vs detección')
ax.legend(fontsize=9)

# Distribución de % sin persona por resultado
ax = axes[1]
neg_sam2 = sam2[sam2['true'] == 0]
tn = neg_sam2[neg_sam2['pred'] == 0]
fp = neg_sam2[neg_sam2['pred'] == 1]
ax.hist(tn['no_person_pct'], bins=15, alpha=0.7, color='#3498db',
        label=f'TN ({len(tn)})')
ax.hist(fp['no_person_pct'], bins=15, alpha=0.7, color='#e74c3c',
        label=f'FP ({len(fp)})')
ax.set_xlabel('% frames sin persona detectada')
ax.set_ylabel('Nº de clips')
ax.set_title('Clips negativos — % fallback por resultado')
ax.legend(fontsize=9)

plt.suptitle('SAM2: impacto del fallback (frames sin persona)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig7_sam2_nopersona.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Figura 8 — SAM2 vs A: comparativa de det_frames en negativos (FP)

In [ ]:
# Comparar cuántos frames activa cada config en clips negativos
neg_a    = dfs['A'][dfs['A']['true'] == 0].copy()
neg_sam2 = dfs['SAM2'][dfs['SAM2']['true'] == 0].copy()
merged   = neg_a[['clip','category','det_frames','pred']].merge(
    neg_sam2[['clip','det_frames','pred']].rename(
        columns={'det_frames':'det_frames_sam2','pred':'pred_sam2'}),
    on='clip'
)
merged['det_frames'] = merged['det_frames'].astype(int)
merged['det_frames_sam2'] = merged['det_frames_sam2'].astype(int)

fig, ax = plt.subplots(figsize=(10, 6))
colors = merged.apply(
    lambda r: '#e74c3c' if r['pred_sam2']==1 else '#3498db', axis=1
)
ax.scatter(merged['det_frames'], merged['det_frames_sam2'],
           c=colors, alpha=0.6, s=40)
ax.axhline(5, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='umbral=5')
ax.axvline(5, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.set_xlabel('Frames con arma — Config A')
ax.set_ylabel('Frames con arma — Config SAM2')
ax.set_title('Clips negativos: activaciones A vs SAM2\n(rojo=FP en SAM2, azul=TN en SAM2)')
fp_patch = mpatches.Patch(color='#e74c3c', alpha=0.7, label='FP en SAM2')
tn_patch = mpatches.Patch(color='#3498db', alpha=0.7, label='TN en SAM2')
ax.legend(handles=[fp_patch, tn_patch], fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig8_scatter_neg_activaciones.png', dpi=150, bbox_inches='tight')
plt.show()

# Cuántos FP nuevos introduce SAM2 que A no tenía
nuevos_fp = merged[(merged['pred']==0) & (merged['pred_sam2']==1)]
print(f'  FP nuevos que SAM2 introduce (que A no tenía): {len(nuevos_fp)}')
print(f'  Categorías:')
print(nuevos_fp['category'].value_counts().to_string())

---
## 10. Síntesis y conclusiones

In [ ]:
print('='*65)
print('TABLA COMPARATIVA FINAL')
print('='*65)
print(f'  {"Config":<28} {"F1":>7} {"Prec":>7} {"Rec":>7} {"FP":>5} {"FN":>5}')
print('  ' + '-'*60)
for cfg in cfgs:
    m = METRICS[cfg]
    marker = ' ← baseline' if cfg=='A' else ''
    print(f'  {CFG_LABELS[cfg]:<28} {m["F1"]:>7.4f} '
          f'{m["Precision"]:>7.4f} {m["Recall"]:>7.4f} '
          f'{m["FP"]:>5} {m["FN"]:>5}{marker}')

---

### Conclusiones

**1. Ninguna estrategia de segmentación mejora el F1 del baseline (Config A).**  
El baseline con frame completo obtiene F1=0.7949. C10 empata prácticamente (0.7947) y SAM2 cae a 0.7294.

**2. SAM2 dispara los falsos positivos (+58% respecto a A).**  
Los FP pasan de 48 (A) a 76 (SAM2). SAM2 es la peor configuración en 10 de las 12 categorías negativas, incluyendo N3 (Running) que tenía 0% de FP con A y C10 y sube al 62.5% con SAM2.

**3. La causa es la pérdida de contexto por aislamiento excesivo.**  
Al aislar objetos individualmente, el weapon_model pierde el contexto que necesitaba para discriminar armas de objetos cotidianos. Una botella, un teléfono o una mano aislados se parecen más a un arma que cuando aparecen en el frame completo con la persona.

**4. SAM2 mejora el mAP@50 a nivel de frame (0.8354 vs 0.7789).**  
Cuando SAM2 detecta correctamente el arma y la aísla, el bounding box resultante es más preciso. Pero esta mejora en localización no se traduce en mejor clasificación de clip porque los FP introducidos compensan con creces.

**5. SAM2 sí reduce FP en N9 (Phone recording 2h).**  
De 77.8% baja a 66.7% — la única categoría donde SAM2 mejora. Posiblemente porque al aislar el teléfono en posición vertical, el weapon_model lo reconoce mejor como no-arma.

**6. Conclusión general sobre segmentación.**  
La hipótesis de que reducir el espacio de búsqueda mejora la detección no se cumple cuando el detector ya fue entrenado con negativos difíciles (Modelo B con hard negatives de COCO). La segmentación ayudaría si se reentrenara el detector con imágenes preprocesadas del mismo tipo — lo que queda documentado como línea de trabajo futuro.

**Decisión:** Config A (frame completo, sin segmentación) permanece como la configuración óptima de la Stage 2 del pipeline.